> **SYNTHETIC DATA.** Every series in this notebook is generated with `np.random` (seed 42). Seasonality, spikes and volatility found here are properties of the generator, not of any electricity market. No real market data has been used yet; see the README.

# Energy Market Price Forecasting - Initial Data Exploration
## Exploratory Data Analysis of Electricity Prices

**Author:** Amina Ali Hussein  
**Date:** December 2024  
**Status:** Initial exploration phase

This notebook performs initial exploratory analysis on electricity price data to understand:
- Price distributions and summary statistics
- Temporal patterns (seasonality, trends)
- Volatility characteristics
- Potential relationships with exogenous variables

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

## 1. Data Loading

For this initial exploration, we'll create synthetic data that mimics real electricity price patterns.  
**Next steps:** Replace with actual EIA data from https://www.eia.gov/electricity/

In [ ]:
# Generate synthetic electricity price data for demonstration
# This simulates realistic patterns we'd see in actual data

np.random.seed(42)
date_range = pd.date_range(start='2019-01-01', end='2024-12-01', freq='D')
n_days = len(date_range)

# Create base price with trend and seasonality
base_price = 35  # $/MWh baseline
trend = np.linspace(0, 10, n_days)  # Slight upward trend
seasonality = 15 * np.sin(2 * np.pi * np.arange(n_days) / 365.25)  # Annual cycle
weekly_pattern = 5 * np.sin(2 * np.pi * np.arange(n_days) / 7)  # Weekly cycle
noise = np.random.normal(0, 8, n_days)  # Random volatility

# Occasional price spikes (exogenous shocks)
spikes = np.zeros(n_days)
spike_indices = np.random.choice(n_days, size=int(n_days * 0.02), replace=False)
spikes[spike_indices] = np.random.uniform(20, 80, len(spike_indices))

# Combine components
prices = base_price + trend + seasonality + weekly_pattern + noise + spikes
prices = np.maximum(prices, 10)  # Floor at $10/MWh

# Create DataFrame
df = pd.DataFrame({
    'date': date_range,
    'price': prices
})

# Add derived features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.dayofweek
df['quarter'] = df['date'].dt.quarter

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head(10)

## 2. Summary Statistics

In [ ]:
# Basic statistics
print("=" * 50)
print("ELECTRICITY PRICE SUMMARY STATISTICS")
print("=" * 50)
print(df['price'].describe())
print(f"\nCoefficient of Variation: {df['price'].std() / df['price'].mean():.2%}")
print(f"Skewness: {df['price'].skew():.3f}")
print(f"Kurtosis: {df['price'].kurtosis():.3f}")

## 3. Price Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['price'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(df['price'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: ${df["price"].mean():.2f}')
axes[0].axvline(df['price'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: ${df["price"].median():.2f}')
axes[0].set_xlabel('Price ($/MWh)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Electricity Prices', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Box plot by year
df.boxplot(column='price', by='year', ax=axes[1])
axes[1].set_xlabel('Year', fontsize=12)
axes[1].set_ylabel('Price ($/MWh)', fontsize=12)
axes[1].set_title('Price Distribution by Year', fontsize=14, fontweight='bold')
plt.suptitle('')  # Remove automatic title

plt.tight_layout()
plt.show()

## 4. Time Series Visualization

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Full time series
axes[0].plot(df['date'], df['price'], linewidth=1, alpha=0.7)
axes[0].set_xlabel('Date', fontsize=12)
axes[0].set_ylabel('Price ($/MWh)', fontsize=12)
axes[0].set_title('Electricity Prices Over Time (2019-2024)', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)

# Zoom into recent period (last 6 months)
recent_df = df[df['date'] >= '2024-06-01']
axes[1].plot(recent_df['date'], recent_df['price'], linewidth=1.5, color='darkblue')
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('Price ($/MWh)', fontsize=12)
axes[1].set_title('Recent Price Movements (Last 6 Months)', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Seasonality Analysis

In [ ]:
# Monthly average prices
monthly_avg = df.groupby('month')['price'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Monthly seasonality
axes[0].bar(monthly_avg['month'], monthly_avg['price'], color='steelblue', edgecolor='black')
axes[0].set_xlabel('Month', fontsize=12)
axes[0].set_ylabel('Average Price ($/MWh)', fontsize=12)
axes[0].set_title('Average Electricity Price by Month', fontsize=14, fontweight='bold')
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
axes[0].grid(alpha=0.3, axis='y')

# Day of week pattern
dow_avg = df.groupby('day_of_week')['price'].mean().reset_index()
axes[1].plot(dow_avg['day_of_week'], dow_avg['price'], marker='o', linewidth=2, markersize=8, color='darkgreen')
axes[1].set_xlabel('Day of Week', fontsize=12)
axes[1].set_ylabel('Average Price ($/MWh)', fontsize=12)
axes[1].set_title('Average Price by Day of Week', fontsize=14, fontweight='bold')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nObservations:")
print(f"- Highest average price month: {monthly_avg.loc[monthly_avg['price'].idxmax(), 'month']}")
print(f"- Lowest average price month: {monthly_avg.loc[monthly_avg['price'].idxmin(), 'month']}")
print(f"- Peak-to-trough monthly difference: ${monthly_avg['price'].max() - monthly_avg['price'].min():.2f}/MWh")

## 6. Volatility Analysis

In [ ]:
# Calculate rolling statistics
df['rolling_mean_30d'] = df['price'].rolling(window=30).mean()
df['rolling_std_30d'] = df['price'].rolling(window=30).std()

fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Price with rolling mean
axes[0].plot(df['date'], df['price'], alpha=0.4, label='Daily Price', linewidth=0.8)
axes[0].plot(df['date'], df['rolling_mean_30d'], color='red', linewidth=2, label='30-Day Rolling Mean')
axes[0].set_xlabel('Date', fontsize=12)
axes[0].set_ylabel('Price ($/MWh)', fontsize=12)
axes[0].set_title('Price with 30-Day Rolling Average', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Rolling volatility
axes[1].plot(df['date'], df['rolling_std_30d'], color='darkred', linewidth=1.5)
axes[1].fill_between(df['date'], 0, df['rolling_std_30d'], alpha=0.3, color='red')
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('Standard Deviation ($/MWh)', fontsize=12)
axes[1].set_title('30-Day Rolling Volatility', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Identifying Price Spikes (Potential Exogenous Shocks)

In [ ]:
# Define spike threshold (e.g., 2 standard deviations above mean)
threshold = df['price'].mean() + 2 * df['price'].std()
spikes_df = df[df['price'] > threshold].copy()

print(f"Price spike threshold: ${threshold:.2f}/MWh")
print(f"Number of spike days: {len(spikes_df)} ({len(spikes_df)/len(df)*100:.1f}% of days)")
print(f"\nTop 10 highest price days:")
print(spikes_df.nlargest(10, 'price')[['date', 'price']])

# Visualize spikes
plt.figure(figsize=(15, 6))
plt.plot(df['date'], df['price'], alpha=0.5, linewidth=1, label='Daily Price')
plt.scatter(spikes_df['date'], spikes_df['price'], color='red', s=50, alpha=0.7, label='Price Spikes', zorder=5)
plt.axhline(threshold, color='orange', linestyle='--', linewidth=2, label=f'Spike Threshold: ${threshold:.2f}')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price ($/MWh)', fontsize=12)
plt.title('Electricity Prices with Identified Spikes', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Next Steps

### Immediate Actions:
1. **Obtain real data** from EIA and NOAA
2. **Integrate weather variables** (temperature, precipitation, wind speed)
3. **Add natural gas prices** (strong correlation with electricity)
4. **Create lagged features** for time series modeling

### Modeling Pipeline:
1. Train/test split (80/20, preserving temporal order)
2. Baseline models (naive, ARIMA, SARIMAX)
3. Machine learning models (XGBoost, Random Forest)
4. Deep learning (LSTM for sequence modeling)
5. Ensemble methods combining best performers

### Analysis Questions to Address:
- How do extreme weather events affect price volatility?
- What is the lead time for predictable price movements?
- Can we identify early warning signals for price spikes?
- How do different forecasting horizons (1-day, 7-day, 30-day) compare in accuracy?

---
**Author:** Amina Ali Hussein  
**Contact:** aminaali9816@gmail.com  
**Last Updated:** December 2024